# School Uniform Detector Training - YOLOv8

This notebook prepares the school-uniform dataset, fine-tunes `yolov8s.pt`, validates on a clean validation split, and builds a Windows 11 deployment package. The workflow uses only `train` and `val`; no test split is generated.

## 1. Environment and Configuration

In [1]:
# Cell 1 - Environment and fixed configuration
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ.setdefault("PYTHONUTF8", "1")

IMAGE_DATASET_DIR = Path("/content/drive/MyDrive/DATN2/dataset_images")
LABELS_ALL_DIR = Path("/content/drive/MyDrive/DATN2/labels/all")
OUTPUT_ROOT = Path("/content/drive/MyDrive/DATN2/yolov8_uniform_training_output")
WORK_DIR = Path("/content/uniform_yolo_dataset")
DATA_YAML_PATH = WORK_DIR / "data.yaml"
FRAMEWORK_LOG_DIR = OUTPUT_ROOT / "framework_logs"
ZIP_PATH = Path("/content/drive/MyDrive/DATN2/yolov8_uniform_windows_package.zip")

CLASS_NAMES = [
    "ao_so_mi_trang",
    "ao_doan_thanh_nien",
    "quan_tay_dai_den",
    "khan_quang_do",
    "quan_short_tay_den",
    "quan_dai_trang",
]
NUM_CLASSES = len(CLASS_NAMES)

BAD_IDS = {357, 4105}
ALLOWED_LEGACY_IDS = {1193, 1194, 2329}
TRAIN_ONLY_ID_RANGE = range(4904, 5301)
RANDOM_SEED = 42
VALIDATION_RATIO = 0.10

MODEL_WEIGHTS = "yolov8s.pt"
RUN_NAME = "uniform_detector_training"
VALIDATION_RUN_NAME = "uniform_detector_validation"
IMG_SIZE = 640
EPOCHS = 200
BATCH_SIZE = 8
WORKERS = 2
PATIENCE = 40
CACHE = False
PREVIEW_SAMPLES_PER_SPLIT = 8
SAMPLE_PREDICTION_COUNT = 8

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FRAMEWORK_LOG_DIR.mkdir(parents=True, exist_ok=True)

install_log = FRAMEWORK_LOG_DIR / "package_install.log"
required_packages = ["ultralytics", "pandas", "pyyaml", "pillow", "matplotlib"]
process = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *required_packages],
    text=True,
    capture_output=True,
)
install_log.write_text("STDOUT\n" + process.stdout + "\nSTDERR\n" + process.stderr, encoding="utf-8")
if process.returncode != 0:
    raise RuntimeError(f"Package installation failed. See {install_log}")

import torch

print("Environment prepared.")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Mounted at /content/drive
Environment prepared.
CUDA available: True
GPU: Tesla T4


## 2. Dataset and Logging Utilities

In [2]:
# Cell 2 - Shared utilities for scanning, splitting, reporting, logging, and packaging
from __future__ import annotations

import contextlib
import hashlib
import io
import json
import logging
import math
import random
import re
import shutil
import traceback
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
import yaml

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
ID_PATTERN = re.compile(r"id(\d+)(?:_|$)")
INTEGER_PATTERN = re.compile(r"^[+-]?\d+$")
BOUNDARY_TOLERANCE = 1e-6


def json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, range):
        return [value.start, value.stop - 1]
    if isinstance(value, set):
        return sorted(value)
    if hasattr(value, "item"):
        return value.item()
    return str(value)


def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=json_default), encoding="utf-8")


def parse_image_id_from_stem(stem: str) -> int | None:
    match = ID_PATTERN.search(stem)
    return int(match.group(1)) if match else None


def stable_score(record: dict[str, Any], seed: int) -> int:
    key = f"{seed}:{record['image_stem']}".encode("utf-8")
    return int.from_bytes(hashlib.sha256(key).digest()[:8], "big")


def list_source_images(image_dir: Path) -> list[Path]:
    if not image_dir.is_dir():
        raise FileNotFoundError(f"Image source directory not found: {image_dir}")
    return sorted(path for path in image_dir.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)


def detect_duplicate_image_stems(image_paths: list[Path]) -> dict[str, list[str]]:
    by_stem: dict[str, list[str]] = defaultdict(list)
    for image_path in image_paths:
        by_stem[image_path.stem.casefold()].append(str(image_path))
    return {stem: paths for stem, paths in by_stem.items() if len(paths) > 1}


def inspect_reference_classes(reference_candidates: list[Path], expected_classes: list[str]) -> dict[str, Any]:
    report = {
        "reference_file_found": False,
        "reference_file_path": "",
        "reference_classes": [],
        "required_class_order_present": False,
        "extra_reference_classes_ignored_for_training": [],
    }
    for candidate in reference_candidates:
        if candidate.is_file():
            observed = [line.strip() for line in candidate.read_text(encoding="utf-8-sig", errors="replace").splitlines() if line.strip()]
            report.update(
                {
                    "reference_file_found": True,
                    "reference_file_path": str(candidate),
                    "reference_classes": observed,
                    "required_class_order_present": observed[: len(expected_classes)] == expected_classes,
                    "extra_reference_classes_ignored_for_training": observed[len(expected_classes):],
                }
            )
            if not report["required_class_order_present"]:
                raise RuntimeError(f"Reference classes do not begin with the required six-class order: {candidate}")
            return report
    return report


def validate_yolo_label_file(label_path: Path, class_names: list[str]) -> dict[str, Any]:
    try:
        raw_text = label_path.read_text(encoding="utf-8-sig", errors="replace")
    except OSError as exc:
        return {"is_empty": False, "is_valid": False, "boxes": [], "errors": [f"could not read label file: {exc}"]}

    non_empty_lines = [(idx, line.strip()) for idx, line in enumerate(raw_text.splitlines(), start=1) if line.strip()]
    if not non_empty_lines:
        return {"is_empty": True, "is_valid": True, "boxes": [], "errors": []}

    errors: list[str] = []
    boxes: list[dict[str, Any]] = []
    for line_number, line in non_empty_lines:
        row_errors: list[str] = []
        parts = line.split()
        if len(parts) != 5:
            row_errors.append(f"line {line_number}: expected exactly 5 values, got {len(parts)}")
        if len(parts) == 5:
            class_token, *coord_tokens = parts
            class_id = None
            if not INTEGER_PATTERN.fullmatch(class_token):
                row_errors.append(f"line {line_number}: class_id is not an integer: {class_token!r}")
            else:
                class_id = int(class_token)
                if class_id not in range(len(class_names)):
                    row_errors.append(f"line {line_number}: class_id {class_id} outside 0-{len(class_names) - 1}")
            coords = None
            try:
                coords = tuple(float(value) for value in coord_tokens)
            except ValueError:
                row_errors.append(f"line {line_number}: coordinates are not numeric")
            if coords is not None:
                x_center, y_center, width, height = coords
                if not all(math.isfinite(value) for value in coords):
                    row_errors.append(f"line {line_number}: coordinates must be finite")
                if not (0.0 <= x_center <= 1.0 and 0.0 <= y_center <= 1.0):
                    row_errors.append(f"line {line_number}: x_center and y_center must be in [0, 1]")
                if not (0.0 < width <= 1.0 and 0.0 < height <= 1.0):
                    row_errors.append(f"line {line_number}: width and height must be > 0 and <= 1")
                left = x_center - width / 2.0
                right = x_center + width / 2.0
                top = y_center - height / 2.0
                bottom = y_center + height / 2.0
                if left < -BOUNDARY_TOLERANCE or top < -BOUNDARY_TOLERANCE or right > 1.0 + BOUNDARY_TOLERANCE or bottom > 1.0 + BOUNDARY_TOLERANCE:
                    row_errors.append(f"line {line_number}: bounding box extends outside normalized image boundaries")
            if not row_errors and coords is not None and class_id is not None:
                x_center, y_center, width, height = coords
                boxes.append({"class_id": int(class_id), "class_name": class_names[int(class_id)], "x_center": x_center, "y_center": y_center, "width": width, "height": height})
        errors.extend(row_errors)
    return {"is_empty": False, "is_valid": len(errors) == 0, "boxes": boxes, "errors": errors}


def count_classes(boxes: list[dict[str, Any]], class_names: list[str]) -> dict[str, int]:
    counts = {class_name: 0 for class_name in class_names}
    for box in boxes:
        counts[class_names[int(box["class_id"])]] += 1
    return counts


def make_record(image_path: Path, label_path: Path, parsed_id: int | None, status: str, split_eligibility: str, validation_errors: list[str] | None, object_count: int, class_ids_present: list[int] | None, class_counts: dict[str, int], split_eligibility_reasons: list[str] | None = None) -> dict[str, Any]:
    return {
        "source_image_path": str(image_path),
        "expected_label_path": str(label_path),
        "image_file_name": image_path.name,
        "image_stem": image_path.stem,
        "image_extension": image_path.suffix.lower(),
        "parsed_id": parsed_id,
        "original_record_status": status,
        "split_eligibility": split_eligibility,
        "split_eligibility_reasons": split_eligibility_reasons or [split_eligibility],
        "assigned_split": "",
        "validation_errors": validation_errors or [],
        "object_count": int(object_count),
        "class_ids_present": sorted(class_ids_present or []),
        "class_counts": class_counts,
    }


def scan_dataset(image_dir: Path, labels_all_dir: Path, output_root: Path, class_names: list[str], bad_ids: set[int], train_only_id_range: range) -> dict[str, Any]:
    output_root.mkdir(parents=True, exist_ok=True)
    if not labels_all_dir.is_dir():
        raise FileNotFoundError(f"Annotation source directory not found: {labels_all_dir}")
    image_paths = list_source_images(image_dir)
    duplicate_stems = detect_duplicate_image_stems(image_paths)
    if duplicate_stems:
        conflict_path = output_root / "duplicate_image_stem_conflicts.json"
        write_json(conflict_path, duplicate_stems)
        raise RuntimeError(f"Duplicate image stems make same-stem label matching ambiguous. See {conflict_path}")
    label_files = sorted(path for path in labels_all_dir.glob("*.txt") if path.is_file())
    records: list[dict[str, Any]] = []
    status_counts = Counter()
    matching_label_count = 0
    for image_path in image_paths:
        parsed_id = parse_image_id_from_stem(image_path.stem)
        label_path = labels_all_dir / f"{image_path.stem}.txt"
        if label_path.exists():
            matching_label_count += 1
        zero_counts = {class_name: 0 for class_name in class_names}
        if parsed_id in bad_ids:
            status_counts["excluded_bad_id"] += 1
            records.append(make_record(image_path, label_path, parsed_id, "excluded_bad_id", "excluded_bad_id", ["known annotation error ID excluded"], 0, [], zero_counts))
            continue
        if not label_path.exists():
            status_counts["missing_label"] += 1
            records.append(make_record(image_path, label_path, parsed_id, "missing_label", "excluded_missing_label", ["matching same-stem label file not found"], 0, [], zero_counts))
            continue
        validation = validate_yolo_label_file(label_path, class_names)
        boxes = validation["boxes"]
        class_ids_present = sorted({int(box["class_id"]) for box in boxes})
        class_counts = count_classes(boxes, class_names)
        if validation["is_empty"]:
            status_counts["empty_negative_label"] += 1
            reasons = ["train_only_empty_negative"]
            if parsed_id in train_only_id_range:
                reasons.append("excluded_train_only_range")
            records.append(make_record(image_path, label_path, parsed_id, "empty_negative_label", "train_only_empty_negative", [], 0, [], zero_counts, reasons))
            continue
        if not validation["is_valid"]:
            status_counts["invalid_label"] += 1
            records.append(make_record(image_path, label_path, parsed_id, "invalid_label", "excluded_invalid_label", list(validation["errors"]), len(boxes), class_ids_present, class_counts))
            continue
        split_eligibility = "excluded_train_only_range" if parsed_id in train_only_id_range else "eligible_for_train_and_val"
        status_counts["valid_positive"] += 1
        records.append(make_record(image_path, label_path, parsed_id, "valid_positive", split_eligibility, [], len(boxes), class_ids_present, class_counts))
    summary = {
        "total_source_images_found": len(image_paths),
        "total_label_files_in_labels_all": len(label_files),
        "total_matching_label_files_found": matching_label_count,
        "record_status_counts": dict(status_counts),
        "supported_image_extensions": sorted(IMAGE_EXTENSIONS),
    }
    return {"records": records, "summary": summary}



def create_train_val_split(scan_result: dict[str, Any], class_names: list[str], bad_ids: set[int], allowed_legacy_ids: set[int], train_only_id_range: range, seed: int, validation_ratio: float) -> dict[str, Any]:
    records = scan_result["records"]
    expected_class_ids = set(range(len(class_names)))
    for record in records:
        record["assigned_split"] = ""
    usable_records = [record for record in records if record["original_record_status"] in {"valid_positive", "empty_negative_label"}]
    validation_candidates = [record for record in usable_records if record["original_record_status"] == "valid_positive" and record["parsed_id"] not in train_only_id_range]
    target_val_count = int(round(len(usable_records) * validation_ratio))
    if validation_candidates and target_val_count == 0 and len(usable_records) > 1:
        target_val_count = 1
    target_val_count = min(target_val_count, len(validation_candidates))
    selected_val: list[dict[str, Any]] = []
    selected_stems: set[str] = set()
    classes_with_candidates = sorted({class_id for record in validation_candidates for class_id in record["class_ids_present"]})
    unavailable_class_ids = sorted(expected_class_ids - set(classes_with_candidates))
    if target_val_count >= len(classes_with_candidates):
        for class_id in classes_with_candidates:
            choices = [record for record in validation_candidates if class_id in record["class_ids_present"] and record["image_stem"] not in selected_stems]
            if choices:
                chosen = min(choices, key=lambda record: stable_score(record, seed + class_id))
                selected_val.append(chosen)
                selected_stems.add(chosen["image_stem"])
    remaining = [record for record in validation_candidates if record["image_stem"] not in selected_stems]
    remaining.sort(key=lambda record: (stable_score(record, seed), record["image_stem"]))
    for record in remaining:
        if len(selected_val) >= target_val_count:
            break
        selected_val.append(record)
        selected_stems.add(record["image_stem"])
    val_stems = {record["image_stem"] for record in selected_val}
    train_records = [record for record in usable_records if record["image_stem"] not in val_stems]
    val_records = list(selected_val)
    for record in train_records:
        record["assigned_split"] = "train"
    for record in val_records:
        record["assigned_split"] = "val"
    train_stems = {record["image_stem"] for record in train_records}
    assert train_stems.isdisjoint(val_stems), "Train and validation splits overlap."
    assert not any(record["parsed_id"] in bad_ids for record in train_records + val_records), "BAD_IDS leaked into a usable split."
    assert not any(record["original_record_status"] == "missing_label" for record in train_records + val_records), "Missing-label record leaked into a split."
    assert not any(record["original_record_status"] == "invalid_label" for record in train_records + val_records), "Invalid-label record leaked into a split."
    assert not any(record["original_record_status"] == "empty_negative_label" for record in val_records), "Empty negative label leaked into validation."
    assert not any(record["parsed_id"] in train_only_id_range for record in val_records), "Train-only ID range leaked into validation."
    for legacy_id in sorted(allowed_legacy_ids):
        for record in [record for record in records if record["parsed_id"] == legacy_id]:
            assert record["original_record_status"] != "excluded_bad_id", f"Allowed legacy ID {legacy_id} was treated as a bad ID."
            assert record["split_eligibility"] != "excluded_bad_id", f"Allowed legacy ID {legacy_id} was excluded by ID filtering."
            if record["original_record_status"] in {"valid_positive", "empty_negative_label"}:
                assert record["assigned_split"] in {"train", "val"}, f"Allowed legacy ID {legacy_id} was not assigned despite being usable."
    val_classes = {class_id for record in val_records for class_id in record["class_ids_present"]}
    missing_candidate_class_ids = sorted(set(classes_with_candidates) - val_classes)
    if target_val_count >= len(classes_with_candidates):
        assert not missing_candidate_class_ids, f"Validation split is missing classes despite sufficient candidates: {missing_candidate_class_ids}"
    total_usable = len(train_records) + len(val_records)
    summary = {
        "target_validation_images": target_val_count,
        "train_images": len(train_records),
        "validation_images": len(val_records),
        "final_train_percentage": round((len(train_records) / total_usable) * 100, 4) if total_usable else 0.0,
        "final_validation_percentage": round((len(val_records) / total_usable) * 100, 4) if total_usable else 0.0,
        "classes_with_validation_candidates": classes_with_candidates,
        "validation_class_ids_present": sorted(val_classes),
        "validation_class_ids_missing": sorted(expected_class_ids - val_classes),
        "validation_class_ids_unavailable": unavailable_class_ids,
        "validation_class_names_unavailable": [class_names[class_id] for class_id in unavailable_class_ids],
        "validation_candidate_class_ids_missing_from_val": missing_candidate_class_ids,
    }
    return {"train": train_records, "val": val_records, "summary": summary}

def ensure_safe_work_dir(path: Path, allow_non_content_work_dir: bool = False) -> None:
    resolved = path.resolve()
    if resolved in {Path("/"), Path("/content"), Path("/content/drive")}:
        raise RuntimeError(f"Refusing to clean unsafe work directory: {resolved}")
    if not allow_non_content_work_dir and not str(resolved).startswith("/content/"):
        raise RuntimeError(f"Temporary YOLO dataset must stay under /content: {resolved}")


def materialize_yolo_dataset(splits: dict[str, Any], work_dir: Path, class_names: list[str], data_yaml_path: Path, bad_ids: set[int], train_only_id_range: range, allow_non_content_work_dir: bool = False) -> list[dict[str, Any]]:
    ensure_safe_work_dir(work_dir, allow_non_content_work_dir=allow_non_content_work_dir)
    if work_dir.exists():
        shutil.rmtree(work_dir)
    for split_name in ["train", "val"]:
        (work_dir / "images" / split_name).mkdir(parents=True, exist_ok=True)
        (work_dir / "labels" / split_name).mkdir(parents=True, exist_ok=True)
    copied_manifest: list[dict[str, Any]] = []
    for split_name in ["train", "val"]:
        txt_lines: list[str] = []
        for record in splits[split_name]:
            source_image = Path(record["source_image_path"])
            source_label = Path(record["expected_label_path"])
            if not source_image.exists():
                raise FileNotFoundError(f"Selected image is missing: {source_image}")
            if not source_label.exists():
                raise FileNotFoundError(f"Selected label is missing: {source_label}")
            target_image = work_dir / "images" / split_name / source_image.name
            target_label = work_dir / "labels" / split_name / source_label.name
            shutil.copy2(source_image, target_image)
            shutil.copy2(source_label, target_label)
            txt_lines.append(str(target_image))
            copied_manifest.append({"split": split_name, "parsed_id": record["parsed_id"], "image_stem": record["image_stem"], "source_image": str(source_image), "source_label": str(source_label), "target_image": str(target_image), "target_label": str(target_label), "original_record_status": record["original_record_status"], "object_count": record["object_count"]})
        (work_dir / f"{split_name}.txt").write_text("\n".join(txt_lines) + ("\n" if txt_lines else ""), encoding="utf-8")
    (work_dir / "classes.txt").write_text("\n".join(class_names) + "\n", encoding="utf-8")
    data_yaml = {"path": str(work_dir), "train": "images/train", "val": "images/val", "nc": len(class_names), "names": {index: class_name for index, class_name in enumerate(class_names)}}
    data_yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True), encoding="utf-8")
    assert not (work_dir / "images" / "test").exists(), "images/test must not exist."
    assert not (work_dir / "labels" / "test").exists(), "labels/test must not exist."
    assert not (work_dir / "test.txt").exists(), "test.txt must not exist."
    generated_yaml = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8"))
    assert "test" not in generated_yaml, "Generated data.yaml must not contain a test key."
    assert not any(item["parsed_id"] in bad_ids for item in copied_manifest), "BAD_IDS were copied."
    assert not any(item["split"] == "val" and item["parsed_id"] in train_only_id_range for item in copied_manifest), "Train-only ID range copied into validation."
    train_stems = {item["image_stem"] for item in copied_manifest if item["split"] == "train"}
    val_stems = {item["image_stem"] for item in copied_manifest if item["split"] == "val"}
    assert train_stems.isdisjoint(val_stems), "Copied train and validation sets overlap."
    for item in copied_manifest:
        assert Path(item["target_image"]).exists(), f"Copied image missing: {item['target_image']}"
        assert Path(item["target_label"]).exists(), f"Copied label missing: {item['target_label']}"
        if item["original_record_status"] == "empty_negative_label":
            assert Path(item["target_label"]).read_text(encoding="utf-8-sig", errors="replace").strip() == "", "Copied empty label did not remain empty."
    return copied_manifest


def flatten_record_for_csv(record: dict[str, Any], class_names: list[str]) -> dict[str, Any]:
    flat = {"source_image_path": record["source_image_path"], "expected_label_path": record["expected_label_path"], "image_stem": record["image_stem"], "parsed_id": record["parsed_id"], "original_record_status": record["original_record_status"], "split_eligibility": record["split_eligibility"], "split_eligibility_reasons": json.dumps(record.get("split_eligibility_reasons", []), ensure_ascii=False), "assigned_split": record["assigned_split"], "validation_errors": json.dumps(record["validation_errors"], ensure_ascii=False), "object_count": record["object_count"], "class_ids_present": json.dumps(record["class_ids_present"], ensure_ascii=False)}
    for class_name in class_names:
        flat[f"objects_{class_name}"] = int(record["class_counts"].get(class_name, 0))
    return flat


def object_counts_by_class(records: list[dict[str, Any]], class_names: list[str]) -> dict[str, int]:
    counts = {class_name: 0 for class_name in class_names}
    for record in records:
        for class_name in class_names:
            counts[class_name] += int(record["class_counts"].get(class_name, 0))
    return counts


def export_dataset_reports(scan_result: dict[str, Any], splits: dict[str, Any], copied_manifest: list[dict[str, Any]], output_root: Path, class_names: list[str], config: dict[str, Any], reference_class_report: dict[str, Any] | None = None) -> dict[str, Path]:
    output_root.mkdir(parents=True, exist_ok=True)
    records = scan_result["records"]
    train_records = splits["train"]
    val_records = splits["val"]
    usable_total = len(train_records) + len(val_records)
    status_counter = Counter(record["original_record_status"] for record in records)
    train_class_counts = object_counts_by_class(train_records, class_names)
    val_class_counts = object_counts_by_class(val_records, class_names)
    dataset_overview = {
        "total_source_images_found": scan_result["summary"]["total_source_images_found"],
        "total_matching_label_files_found": scan_result["summary"]["total_matching_label_files_found"],
        "total_label_files_in_labels_all": scan_result["summary"]["total_label_files_in_labels_all"],
        "valid_positive_images": int(status_counter.get("valid_positive", 0)),
        "empty_negative_label_images": int(status_counter.get("empty_negative_label", 0)),
        "missing_label_images": int(status_counter.get("missing_label", 0)),
        "invalid_label_images": int(status_counter.get("invalid_label", 0)),
        "excluded_bad_id_images": int(status_counter.get("excluded_bad_id", 0)),
        "train_images": len(train_records),
        "validation_images": len(val_records),
        "train_positive_images": sum(1 for record in train_records if record["original_record_status"] == "valid_positive"),
        "train_empty_negative_images": sum(1 for record in train_records if record["original_record_status"] == "empty_negative_label"),
        "validation_positive_images": sum(1 for record in val_records if record["original_record_status"] == "valid_positive"),
        "object_instances_per_class_train": train_class_counts,
        "object_instances_per_class_val": val_class_counts,
        "final_train_percentage": round((len(train_records) / usable_total) * 100, 4) if usable_total else 0.0,
        "final_validation_percentage": round((len(val_records) / usable_total) * 100, 4) if usable_total else 0.0,
        "random_seed": config["random_seed"],
        "bad_ids": sorted(config["bad_ids"]),
        "allowed_legacy_ids": sorted(config["allowed_legacy_ids"]),
        "legacy_ids_are_eligible_for_normal_split_processing": True,
        "train_only_id_range": [config["train_only_id_range"].start, config["train_only_id_range"].stop - 1],
        "model_weights": config["model_weights"],
        "no_test_split_created": True,
        "nc": len(class_names),
        "class_names": {index: class_name for index, class_name in enumerate(class_names)},
        "validation_class_ids_present": splits["summary"].get("validation_class_ids_present", []),
        "validation_class_ids_missing": splits["summary"].get("validation_class_ids_missing", []),
        "validation_class_ids_unavailable": splits["summary"].get("validation_class_ids_unavailable", []),
        "validation_class_names_unavailable": splits["summary"].get("validation_class_names_unavailable", []),
        "validation_candidate_class_ids_missing_from_val": splits["summary"].get("validation_candidate_class_ids_missing_from_val", []),
        "reference_classes_check": reference_class_report or {},
    }
    paths = {
        "dataset_validation_report_json": output_root / "dataset_validation_report.json",
        "dataset_validation_report_csv": output_root / "dataset_validation_report.csv",
        "split_summary_csv": output_root / "split_summary.csv",
        "split_manifest_csv": output_root / "split_manifest.csv",
        "class_distribution_csv": output_root / "class_distribution.csv",
        "class_distribution_png": output_root / "class_distribution.png",
        "dataset_overview_json": output_root / "dataset_overview.json",
        "training_configuration_json": output_root / "training_configuration.json",
    }
    write_json(paths["dataset_validation_report_json"], {"summary": dataset_overview, "records": records})
    pd.DataFrame([flatten_record_for_csv(record, class_names) for record in records]).to_csv(paths["dataset_validation_report_csv"], index=False, encoding="utf-8-sig")
    split_summary_rows = []
    for split_name, split_records in [("train", train_records), ("val", val_records)]:
        split_summary_rows.append({"split": split_name, "images": len(split_records), "labels": len(split_records), "positive_images": sum(1 for record in split_records if record["original_record_status"] == "valid_positive"), "empty_negative_images": sum(1 for record in split_records if record["original_record_status"] == "empty_negative_label"), "object_instances": sum(record["object_count"] for record in split_records), "percentage_of_usable_dataset": round((len(split_records) / usable_total) * 100, 4) if usable_total else 0.0})
    pd.DataFrame(split_summary_rows).to_csv(paths["split_summary_csv"], index=False, encoding="utf-8-sig")
    split_manifest_rows = []
    for split_name, split_records in [("train", train_records), ("val", val_records)]:
        for record in split_records:
            split_manifest_rows.append({"split": split_name, "image_stem": record["image_stem"], "parsed_id": record["parsed_id"], "source_image_path": record["source_image_path"], "expected_label_path": record["expected_label_path"], "original_record_status": record["original_record_status"], "split_eligibility": record["split_eligibility"], "object_count": record["object_count"], "class_ids_present": json.dumps(record["class_ids_present"], ensure_ascii=False)})
    pd.DataFrame(split_manifest_rows).to_csv(paths["split_manifest_csv"], index=False, encoding="utf-8-sig")
    class_distribution_rows = []
    for split_name, split_records in [("train", train_records), ("val", val_records)]:
        counts = object_counts_by_class(split_records, class_names)
        for class_id, class_name in enumerate(class_names):
            class_distribution_rows.append({"split": split_name, "class_id": class_id, "class_name": class_name, "instances": counts[class_name]})
    class_distribution_df = pd.DataFrame(class_distribution_rows)
    class_distribution_df.to_csv(paths["class_distribution_csv"], index=False, encoding="utf-8-sig")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 5))
    x_positions = list(range(len(class_names)))
    width = 0.36
    ax.bar([x - width / 2 for x in x_positions], [train_class_counts[c] for c in class_names], width=width, label="train", color="#2563eb")
    ax.bar([x + width / 2 for x in x_positions], [val_class_counts[c] for c in class_names], width=width, label="val", color="#dc2626")
    ax.set_xticks(x_positions)
    ax.set_xticklabels(class_names, rotation=18, ha="right")
    ax.set_ylabel("Object instances")
    ax.set_title("Class distribution by split")
    ax.legend()
    fig.tight_layout()
    fig.savefig(paths["class_distribution_png"], dpi=180)
    plt.close(fig)
    write_json(paths["dataset_overview_json"], dataset_overview)
    training_configuration = {"image_dataset_dir": str(config["image_dataset_dir"]), "labels_all_dir": str(config["labels_all_dir"]), "output_root": str(config["output_root"]), "temporary_work_dir": str(config["work_dir"]), "data_yaml_path": str(config["data_yaml_path"]), "model_weights": config["model_weights"], "training_method": "transfer learning / fine-tuning from yolov8s.pt", "nc": len(class_names), "class_names": class_names, "bad_ids": sorted(config["bad_ids"]), "allowed_legacy_ids": sorted(config["allowed_legacy_ids"]), "train_only_id_range": [config["train_only_id_range"].start, config["train_only_id_range"].stop - 1], "random_seed": config["random_seed"], "validation_ratio_target": config["validation_ratio"], "imgsz": config["img_size"], "epochs": config["epochs"], "batch": config["batch_size"], "workers": config["workers"], "patience": config["patience"], "cache": config["cache"], "augmentation": {"fliplr": 0.5, "degrees": 7.0, "translate": 0.10, "scale": 0.40, "perspective": 0.0005, "hsv_h": 0.015, "hsv_s": 0.50, "hsv_v": 0.35, "mosaic": 1.0, "cos_lr": True}, "splits": {"train": len(train_records), "val": len(val_records)}}
    write_json(paths["training_configuration_json"], training_configuration)
    assert set(pd.read_csv(paths["split_summary_csv"])["split"]) == {"train", "val"}, "split_summary.csv contains a non train/val split."
    assert set(pd.read_csv(paths["class_distribution_csv"])["split"]) == {"train", "val"}, "class_distribution.csv contains a non train/val split."
    assert set(pd.read_csv(paths["split_manifest_csv"])["split"]).issubset({"train", "val"}), "split_manifest.csv contains a non train/val split."
    return paths



@contextlib.contextmanager
def quiet_execution(log_dir: Path, log_name: str):
    log_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = log_dir / f"{timestamp}_{log_name}.log"
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()
    logger_buffer = io.StringIO()
    handler = logging.StreamHandler(logger_buffer)
    handler.setLevel(logging.DEBUG)
    loggers = [logging.getLogger(), logging.getLogger("ultralytics")]
    old_levels = [logger.level for logger in loggers]
    for logger in loggers:
        logger.addHandler(handler)
    exception_text = ""
    try:
        with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
            yield log_path
    except Exception:
        exception_text = traceback.format_exc()
        raise
    finally:
        for logger, old_level in zip(loggers, old_levels):
            logger.removeHandler(handler)
            logger.setLevel(old_level)
        content = [f"log_name: {log_name}", f"\ncreated_at: {datetime.now().isoformat(timespec='seconds')}", "\n\nSTDOUT\n", stdout_buffer.getvalue(), "\nSTDERR\n", stderr_buffer.getvalue(), "\nLOGGER\n", logger_buffer.getvalue()]
        if exception_text:
            content.extend(["\nEXCEPTION\n", exception_text])
        log_path.write_text("".join(content), encoding="utf-8")


def export_training_curves(results_csv_path: Path, output_path: Path) -> Path:
    if not results_csv_path.exists():
        raise FileNotFoundError(f"results.csv not found: {results_csv_path}")
    results_df = pd.read_csv(results_csv_path)
    results_df.columns = [column.strip() for column in results_df.columns]
    if results_df.empty:
        raise RuntimeError(f"results.csv is empty: {results_csv_path}")
    import matplotlib.pyplot as plt
    x_values = results_df["epoch"] if "epoch" in results_df.columns else results_df.index
    train_loss_cols = [column for column in results_df.columns if column.startswith("train/") and column.endswith("_loss")]
    val_loss_cols = [column for column in results_df.columns if column.startswith("val/") and column.endswith("_loss")]
    metric_cols = [column for column in results_df.columns if "mAP" in column or "precision" in column or "recall" in column]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    if train_loss_cols or val_loss_cols:
        for column in train_loss_cols + val_loss_cols:
            axes[0].plot(x_values, results_df[column], label=column)
        axes[0].set_title("Training and validation losses")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Loss")
        axes[0].legend(fontsize=8)
    else:
        axes[0].text(0.5, 0.5, "No loss columns found", ha="center", va="center")
        axes[0].set_axis_off()
    if metric_cols:
        for column in metric_cols:
            axes[1].plot(x_values, results_df[column], label=column)
        axes[1].set_title("Validation metrics")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Metric")
        axes[1].legend(fontsize=8)
    else:
        axes[1].text(0.5, 0.5, "No metric columns found", ha="center", va="center")
        axes[1].set_axis_off()
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180)
    plt.close(fig)
    return output_path


def summarize_results_csv(results_csv_path: Path) -> dict[str, Any]:
    results_df = pd.read_csv(results_csv_path)
    results_df.columns = [column.strip() for column in results_df.columns]
    if results_df.empty:
        return {"rows": 0}
    last_row = results_df.iloc[-1].to_dict()
    summary = {"rows": len(results_df)}
    for key in ["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
        if key in last_row:
            summary[key] = float(last_row[key])
    return summary


def metric_value(metrics: dict[str, Any], candidates: list[str]) -> float | None:
    for candidate in candidates:
        if candidate in metrics and metrics[candidate] is not None:
            try:
                return float(metrics[candidate])
            except (TypeError, ValueError):
                return None
    return None


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


## 3. Dataset Scan and Validation

In [3]:
# Cell 3 - Scan source images and validate same-stem YOLO labels
REFERENCE_CLASS_REPORT = inspect_reference_classes([Path("classes.txt"), OUTPUT_ROOT.parent / "classes.txt"], CLASS_NAMES)
SCAN_RESULT = scan_dataset(IMAGE_DATASET_DIR, LABELS_ALL_DIR, OUTPUT_ROOT, CLASS_NAMES, BAD_IDS, TRAIN_ONLY_ID_RANGE)
print("Dataset scan completed.")
print(f"Source images found: {SCAN_RESULT['summary']['total_source_images_found']}")
print(f"Matching labels found: {SCAN_RESULT['summary']['total_matching_label_files_found']}")
print(f"Record status counts: {SCAN_RESULT['summary']['record_status_counts']}")
print("Dataset validation completed.")


Dataset scan completed.
Source images found: 3919
Matching labels found: 3919
Record status counts: {'valid_positive': 3506, 'excluded_bad_id': 2, 'empty_negative_label': 411}
Dataset validation completed.


## 4. Train/Validation Split

In [4]:
# Cell 4 - Build deterministic train/val split only
SPLITS = create_train_val_split(SCAN_RESULT, CLASS_NAMES, BAD_IDS, ALLOWED_LEGACY_IDS, TRAIN_ONLY_ID_RANGE, RANDOM_SEED, VALIDATION_RATIO)
print("Train/validation split created.")
print(f"Train images: {SPLITS['summary']['train_images']}")
print(f"Validation images: {SPLITS['summary']['validation_images']}")
print(f"Actual train percentage: {SPLITS['summary']['final_train_percentage']:.2f}%")
print(f"Actual validation percentage: {SPLITS['summary']['final_validation_percentage']:.2f}%")

if SPLITS["summary"].get("validation_class_ids_unavailable"):
    unavailable = SPLITS["summary"]["validation_class_ids_unavailable"]
    names = SPLITS["summary"]["validation_class_names_unavailable"]
    print(f"Warning: validation has no eligible positive candidates for class IDs {unavailable}: {names}.")
if SPLITS["summary"].get("validation_candidate_class_ids_missing_from_val"):
    missing = SPLITS["summary"]["validation_candidate_class_ids_missing_from_val"]
    print(f"Warning: validation split could not represent candidate class IDs {missing} within the target validation size.")


Train/validation split created.
Train images: 3525
Validation images: 392
Actual train percentage: 89.99%
Actual validation percentage: 10.01%


## 5. Temporary YOLO Dataset and Reports

In [5]:
# Cell 5 - Create /content train/val dataset and export audit reports
CONFIG_FOR_REPORTS = {
    "image_dataset_dir": IMAGE_DATASET_DIR,
    "labels_all_dir": LABELS_ALL_DIR,
    "output_root": OUTPUT_ROOT,
    "work_dir": WORK_DIR,
    "data_yaml_path": DATA_YAML_PATH,
    "model_weights": MODEL_WEIGHTS,
    "bad_ids": BAD_IDS,
    "allowed_legacy_ids": ALLOWED_LEGACY_IDS,
    "train_only_id_range": TRAIN_ONLY_ID_RANGE,
    "random_seed": RANDOM_SEED,
    "validation_ratio": VALIDATION_RATIO,
    "img_size": IMG_SIZE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "workers": WORKERS,
    "patience": PATIENCE,
    "cache": CACHE,
}
COPIED_MANIFEST = materialize_yolo_dataset(SPLITS, WORK_DIR, CLASS_NAMES, DATA_YAML_PATH, BAD_IDS, TRAIN_ONLY_ID_RANGE)
REPORT_PATHS = export_dataset_reports(SCAN_RESULT, SPLITS, COPIED_MANIFEST, OUTPUT_ROOT, CLASS_NAMES, CONFIG_FOR_REPORTS, REFERENCE_CLASS_REPORT)
print("Dataset reports exported.")
print(f"Temporary data.yaml: {DATA_YAML_PATH}")
print(f"Split manifest: {REPORT_PATHS['split_manifest_csv']}")
print(f"Dataset overview: {REPORT_PATHS['dataset_overview_json']}")


Dataset reports exported.
Temporary data.yaml: /content/uniform_yolo_dataset/data.yaml
Split manifest: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/split_manifest.csv
Dataset overview: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/dataset_overview.json


## 6. Bounding Box Preview Samples

In [6]:
# Cell 6 - Generate train/val bbox preview samples under OUTPUT_ROOT only
from PIL import Image, ImageDraw, ImageFont

CLASS_COLORS = [
    (180, 180, 180),
    (0, 174, 239),
    (100, 100, 100),
    (239, 68, 68),
    (168, 85, 247),
    (245, 158, 11),
]


def readable_text_color(background: tuple[int, int, int]) -> tuple[int, int, int]:
    red, green, blue = background
    luminance = 0.299 * red + 0.587 * green + 0.114 * blue
    return (0, 0, 0) if luminance > 160 else (255, 255, 255)


def yolo_box_to_xyxy(box: dict[str, Any], image_width: int, image_height: int) -> tuple[int, int, int, int]:
    x_center = float(box["x_center"])
    y_center = float(box["y_center"])
    width = float(box["width"])
    height = float(box["height"])
    x_min = round((x_center - width / 2.0) * image_width)
    y_min = round((y_center - height / 2.0) * image_height)
    x_max = round((x_center + width / 2.0) * image_width)
    y_max = round((y_center + height / 2.0) * image_height)
    return (max(0, min(image_width - 1, x_min)), max(0, min(image_height - 1, y_min)), max(0, min(image_width - 1, x_max)), max(0, min(image_height - 1, y_max)))


def load_preview_boxes(label_path: Path) -> list[dict[str, Any]]:
    validation = validate_yolo_label_file(label_path, CLASS_NAMES)
    return validation["boxes"] if validation["is_valid"] and not validation["is_empty"] else []


def draw_preview(record: dict[str, Any], output_path: Path) -> None:
    image_path = Path(record["source_image_path"])
    label_path = Path(record["expected_label_path"])
    boxes = load_preview_boxes(label_path)
    with Image.open(image_path) as original:
        image = original.convert("RGB")
        width, height = image.size
        draw = ImageDraw.Draw(image)
        font = ImageFont.load_default()
        line_width = max(2, min(8, round(min(width, height) / 220)))
        if not boxes:
            draw.text((16, 16), "negative sample: no boxes", fill=(17, 24, 39), font=font)
        for box in boxes:
            class_id = int(box["class_id"])
            color = CLASS_COLORS[class_id]
            x_min, y_min, x_max, y_max = yolo_box_to_xyxy(box, width, height)
            for offset in range(line_width):
                draw.rectangle((x_min - offset, y_min - offset, x_max + offset, y_max + offset), outline=color)
            label = CLASS_NAMES[class_id]
            text_bbox = draw.textbbox((0, 0), label, font=font)
            text_width = text_bbox[2] - text_bbox[0]
            text_height = text_bbox[3] - text_bbox[1]
            text_y = max(0, y_min - text_height - 8)
            draw.rectangle((x_min, text_y, x_min + text_width + 8, text_y + text_height + 6), fill=color)
            draw.text((x_min + 4, text_y + 3), label, fill=readable_text_color(color), font=font)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        save_kwargs = {"quality": 95} if output_path.suffix.lower() in {".jpg", ".jpeg"} else {}
        image.save(output_path, **save_kwargs)

preview_root = OUTPUT_ROOT / "bbox_preview_samples"
if preview_root.exists():
    shutil.rmtree(preview_root)
preview_root.mkdir(parents=True, exist_ok=True)
created_previews: list[Path] = []
for split_name in ["train", "val"]:
    records = list(SPLITS[split_name])
    positive_records = [record for record in records if record["object_count"] > 0]
    empty_records = [record for record in records if record["original_record_status"] == "empty_negative_label"]
    selected: list[dict[str, Any]] = []
    if split_name == "train" and empty_records:
        selected.append(min(empty_records, key=lambda record: stable_score(record, RANDOM_SEED)))
    remaining_slots = max(0, PREVIEW_SAMPLES_PER_SPLIT - len(selected))
    candidate_pool = positive_records or records
    selected_stems = {item["image_stem"] for item in selected}
    candidate_pool = [record for record in candidate_pool if record["image_stem"] not in selected_stems]
    candidate_pool.sort(key=lambda record: stable_score(record, RANDOM_SEED + len(split_name)))
    selected.extend(candidate_pool[:remaining_slots])
    for record in selected:
        output_path = preview_root / split_name / Path(record["source_image_path"]).name
        draw_preview(record, output_path)
        created_previews.append(output_path)
print("BBox preview samples exported.")
print(f"Preview images: {len(created_previews)}")
print(f"Preview folder: {preview_root}")


BBox preview samples exported.
Preview images: 16
Preview folder: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/bbox_preview_samples


# Cell 7A - Restore the latest best.pt checkpoint from Google Drive

In [7]:
# Cell 7A - Restore the latest best.pt checkpoint from Google Drive
# Run this after Cells 1–6 when the Colab runtime was disconnected.
# Do NOT run the original training Cell 7 again.

from pathlib import Path

OUTPUT_ROOT = Path("/content/drive/MyDrive/DATN2/yolov8_uniform_training_output")
RUNS_ROOT = OUTPUT_ROOT / "runs"

if not OUTPUT_ROOT.exists():
    raise FileNotFoundError(
        f"OUTPUT_ROOT does not exist: {OUTPUT_ROOT}\n"
        "Make sure Google Drive is mounted and the path is correct."
    )

best_files = sorted(
    [
        path
        for path in OUTPUT_ROOT.rglob("best.pt")
        if path.is_file() and path.stat().st_size > 0
    ],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

last_files = sorted(
    [
        path
        for path in OUTPUT_ROOT.rglob("last.pt")
        if path.is_file() and path.stat().st_size > 0
    ],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

print("Best checkpoints found:")
for path in best_files[:5]:
    print(
        f"  {path}\n"
        f"    Size: {path.stat().st_size / 1024 / 1024:.2f} MB | "
        f"Modified: {path.stat().st_mtime}"
    )

print("\nLast checkpoints found:")
for path in last_files[:5]:
    print(
        f"  {path}\n"
        f"    Size: {path.stat().st_size / 1024 / 1024:.2f} MB | "
        f"Modified: {path.stat().st_mtime}"
    )

if not best_files:
    raise FileNotFoundError(
        "No best.pt was found under OUTPUT_ROOT. "
        "The previous training may not have completed or checkpoints were not saved to Google Drive."
    )

BEST_PT = best_files[0]

# Restore a reasonable RUN_DIR for later cells.
# Expected train run directory structure: .../runs/<run_name>/weights/best.pt
if BEST_PT.parent.name == "weights":
    RUN_DIR = BEST_PT.parent.parent
else:
    RUN_DIR = BEST_PT.parent

print("\nCheckpoint restored successfully.")
print(f"BEST_PT: {BEST_PT}")
print(f"RUN_DIR: {RUN_DIR}")

if "DATA_YAML_PATH" not in globals() or not Path(DATA_YAML_PATH).exists():
    raise FileNotFoundError(
        "DATA_YAML_PATH is missing. Run Cells 1–6 first so the temporary "
        "/content/uniform_yolo_dataset/data.yaml is recreated."
    )

if "SPLITS" not in globals() or not SPLITS.get("val"):
    raise RuntimeError(
        "SPLITS['val'] is unavailable. Run Cells 1–6 first before restoring best.pt."
    )

print(f"Validation images restored: {len(SPLITS['val'])}")
print("You can now run the fixed Cell 8 without retraining.")

Best checkpoints found:
  /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/runs/uniform_detector_training/weights/best.pt
    Size: 21.48 MB | Modified: 1782783559.0

Last checkpoints found:
  /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/runs/uniform_detector_training/weights/last.pt
    Size: 21.48 MB | Modified: 1782783559.0

Checkpoint restored successfully.
BEST_PT: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/runs/uniform_detector_training/weights/best.pt
RUN_DIR: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/runs/uniform_detector_training
Validation images restored: 392
You can now run the fixed Cell 8 without retraining.


## 7. Fine-Tune YOLOv8s

In [ ]:
# Cell 7 - Fine-tune yolov8s.pt with captured framework logs
if not DATA_YAML_PATH.exists():
    raise FileNotFoundError(f"Generated data.yaml not found: {DATA_YAML_PATH}")
print("Training started.")
with quiet_execution(FRAMEWORK_LOG_DIR, "training") as TRAINING_LOG_PATH:
    from ultralytics import YOLO
    model = YOLO(MODEL_WEIGHTS)
    TRAIN_RESULTS = model.train(
        data=str(DATA_YAML_PATH), imgsz=IMG_SIZE, epochs=EPOCHS, patience=PATIENCE, batch=BATCH_SIZE, workers=WORKERS,
        project=str(OUTPUT_ROOT / "runs"), name=RUN_NAME, exist_ok=True, cache=CACHE, optimizer="auto", cos_lr=True,
        close_mosaic=10, seed=RANDOM_SEED, deterministic=True, pretrained=True, plots=True, val=True, verbose=False,
        fliplr=0.5, flipud=0.0, degrees=7.0, translate=0.10, scale=0.40, shear=0.0, perspective=0.0005,
        hsv_h=0.015, hsv_s=0.50, hsv_v=0.35, mosaic=1.0,
    )
RUN_DIR = Path(getattr(TRAIN_RESULTS, "save_dir", OUTPUT_ROOT / "runs" / RUN_NAME))
if not RUN_DIR.exists():
    candidates = sorted((OUTPUT_ROOT / "runs").glob("**/weights/best.pt"), key=lambda path: path.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError("No YOLO training run directory with best.pt was found.")
    RUN_DIR = candidates[0].parents[1]
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"
if not BEST_PT.exists():
    raise FileNotFoundError(f"best.pt not found after training: {BEST_PT}")
if not LAST_PT.exists():
    raise FileNotFoundError(f"last.pt not found after training: {LAST_PT}")
print("Best checkpoint verified.")
RESULTS_CSV = RUN_DIR / "results.csv"
if not RESULTS_CSV.exists():
    raise FileNotFoundError(f"Training results.csv not found: {RESULTS_CSV}")
shutil.copy2(RESULTS_CSV, OUTPUT_ROOT / "results.csv")
if (RUN_DIR / "args.yaml").exists():
    shutil.copy2(RUN_DIR / "args.yaml", OUTPUT_ROOT / "training_args.yaml")
TRAINING_CURVES_PATH = export_training_curves(RESULTS_CSV, OUTPUT_ROOT / "training_curves.png")
TRAINING_RESULTS_SUMMARY = summarize_results_csv(RESULTS_CSV)
write_json(OUTPUT_ROOT / "training_run_metadata.json", {"run_dir": str(RUN_DIR), "best_pt": str(BEST_PT), "last_pt": str(LAST_PT), "results_csv": str(RESULTS_CSV), "training_log": str(TRAINING_LOG_PATH), "training_results_summary": TRAINING_RESULTS_SUMMARY})
print("Training completed.")
print(f"Run directory: {RUN_DIR}")
print(f"Raw training log: {TRAINING_LOG_PATH}")
for key, value in TRAINING_RESULTS_SUMMARY.items():
    print(f"{key}: {value}")


Training started.
Best checkpoint verified.
Training completed.
Run directory: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/runs/uniform_detector_training
Raw training log: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/framework_logs/20260629_202403_training.log
rows: 152
metrics/precision(B): 0.93354
metrics/recall(B): 0.94396
metrics/mAP50(B): 0.95761
metrics/mAP50-95(B): 0.73609


## 8. Validation Metrics and Curves

In [8]:
# Cell 8 - Validate best.pt on the generated val split only (non-blocking)
# Replace the entire old Cell 8 with this code.
# This cell never stops only because a chart such as PR_curve.png is unavailable.

from __future__ import annotations

import json
import shutil
import traceback
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

VALIDATION_STARTED_AT = datetime.now()
VALIDATION_LOG_PATH = FRAMEWORK_LOG_DIR / (
    f"{VALIDATION_STARTED_AT.strftime('%Y%m%d_%H%M%S')}_validation.log"
)
VAL_RUN_DIR = OUTPUT_ROOT / "runs" / VALIDATION_RUN_NAME
VAL_RESULTS = None
validation_model = None
warnings: list[str] = []
errors: list[str] = []
artifact_sources: dict[str, str] = {}
artifact_generation: dict[str, str] = {}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FRAMEWORK_LOG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES_RUNTIME = list(globals().get("CLASS_NAMES", []))
NUM_CLASSES_RUNTIME = len(CLASS_NAMES_RUNTIME)
CLASS_NAME_MAP = {
    index: class_name for index, class_name in enumerate(CLASS_NAMES_RUNTIME)
}
BAD_IDS_RUNTIME = set(globals().get("BAD_IDS", {357, 4105}))
TRAIN_ONLY_RANGE_RUNTIME = globals().get("TRAIN_ONLY_ID_RANGE", range(4904, 5301))
SPLITS_RUNTIME = globals().get("SPLITS", {})

validation_records = (
    list(SPLITS_RUNTIME.get("val", []))
    if isinstance(SPLITS_RUNTIME, dict)
    else []
)


def safe_json(path: Path, payload: dict) -> None:
    try:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(
            json.dumps(payload, ensure_ascii=False, indent=2, default=str),
            encoding="utf-8",
        )
    except Exception as exc:
        errors.append(f"Could not write {path.name}: {exc}")


def get_metric(metrics: dict, keys: list[str]) -> float | None:
    for key in keys:
        try:
            if metrics.get(key) is not None:
                return float(metrics[key])
        except (TypeError, ValueError):
            pass
    return None


def count_objects(records: list[dict]) -> dict[str, int]:
    counts = {class_name: 0 for class_name in CLASS_NAMES_RUNTIME}

    for record in records:
        for class_name in CLASS_NAMES_RUNTIME:
            try:
                counts[class_name] += int(
                    record.get("class_counts", {}).get(class_name, 0)
                )
            except (TypeError, ValueError):
                pass

    return counts


def add_dir(directories: list[Path], value) -> None:
    if not value:
        return

    try:
        path = Path(value)
    except (TypeError, ValueError):
        return

    if path not in directories:
        directories.append(path)


def find_artifact(filename: str, directories: list[Path]) -> Path | None:
    # First search known training/validation run directories.
    for directory in directories:
        path = directory / filename
        if path.is_file():
            return path

    # Then search all run folders and select the newest matching file.
    matches: list[Path] = []

    for root in (OUTPUT_ROOT / "runs", OUTPUT_ROOT):
        if not root.exists():
            continue

        try:
            matches.extend(
                path
                for path in root.rglob(filename)
                if path.is_file() and path != OUTPUT_ROOT / filename
            )
        except Exception as exc:
            warnings.append(f"Could not search {root} for {filename}: {exc}")

    try:
        return max(matches, key=lambda path: path.stat().st_mtime) if matches else None
    except Exception as exc:
        warnings.append(f"Could not select {filename}: {exc}")
        return None


def copy_artifact(filename: str, directories: list[Path]) -> bool:
    target = OUTPUT_ROOT / filename

    if target.is_file():
        artifact_sources[filename] = str(target)
        return True

    source = find_artifact(filename, directories)

    if source is None:
        return False

    try:
        shutil.copy2(source, target)
        artifact_sources[filename] = str(source)
        return True
    except Exception as exc:
        warnings.append(f"Could not copy {filename} from {source}: {exc}")
        return False


def render_missing_real_curves(metrics_object) -> None:
    """
    Render only from real curve arrays returned by the validation result.
    No fake metrics and no placeholder chart images are created.
    """
    if metrics_object is None:
        return

    curve_results = getattr(metrics_object, "curves_results", None)

    try:
        curve_results = curve_results() if callable(curve_results) else curve_results
    except Exception as exc:
        warnings.append(f"Could not read real curve arrays: {exc}")
        return

    if not curve_results:
        return

    curve_specs = [
        ("PR_curve.png", "Precision-Recall curve"),
        ("F1_curve.png", "F1-Confidence curve"),
        ("P_curve.png", "Precision-Confidence curve"),
        ("R_curve.png", "Recall-Confidence curve"),
    ]

    try:
        import matplotlib.pyplot as plt
    except Exception as exc:
        warnings.append(f"Could not import matplotlib for curve export: {exc}")
        return

    for index, (filename, fallback_title) in enumerate(curve_specs):
        output_file = OUTPUT_ROOT / filename

        if output_file.is_file() or index >= len(curve_results):
            continue

        try:
            item = curve_results[index]
            x_values, y_values, x_label, y_label = item[:4]

            x_values = np.asarray(x_values, dtype=float).reshape(-1)
            y_values = np.asarray(y_values, dtype=float)

            if x_values.size == 0 or y_values.size == 0:
                warnings.append(f"{filename}: real curve data is empty.")
                continue

            if y_values.ndim == 1:
                curve_matrix = y_values.reshape(1, -1)

            elif y_values.shape[-1] == x_values.size:
                curve_matrix = y_values.reshape(-1, x_values.size)

            elif y_values.shape[0] == x_values.size:
                curve_matrix = y_values.T.reshape(-1, x_values.size)

            else:
                warnings.append(
                    f"{filename}: curve shape {tuple(y_values.shape)} does not match "
                    f"x-axis length {x_values.size}."
                )
                continue

            curve_matrix = curve_matrix[np.isfinite(curve_matrix).all(axis=1)]

            if curve_matrix.size == 0:
                warnings.append(f"{filename}: no finite real curve values are available.")
                continue

            fig, ax = plt.subplots(figsize=(8, 6))

            for class_index in range(
                min(len(CLASS_NAMES_RUNTIME), curve_matrix.shape[0])
            ):
                ax.plot(
                    x_values,
                    curve_matrix[class_index],
                    linewidth=1.0,
                    alpha=0.45,
                    label=CLASS_NAMES_RUNTIME[class_index],
                )

            ax.plot(
                x_values,
                np.nanmean(curve_matrix, axis=0),
                linewidth=2.5,
                label="mean across available classes",
            )

            curve_titles = getattr(
                metrics_object,
                "curves",
                [fallback_title] * len(curve_specs),
            )

            ax.set_title(str(curve_titles[index]))
            ax.set_xlabel(str(x_label))
            ax.set_ylabel(str(y_label))
            ax.grid(True, alpha=0.25)
            ax.legend(fontsize=8, loc="best")

            fig.tight_layout()
            fig.savefig(output_file, dpi=180)
            plt.close(fig)

            artifact_generation[filename] = (
                "Rendered from real validation curve arrays."
            )

        except Exception as exc:
            warnings.append(f"Could not render {filename} from real metrics: {exc}")


# ------------------------------------------------------------------
# Pre-validation checks: record warnings, do not interrupt the cell.
# ------------------------------------------------------------------
precheck_errors: list[str] = []

best_pt_value = globals().get("BEST_PT")
data_yaml_value = globals().get("DATA_YAML_PATH")

if not best_pt_value or not Path(best_pt_value).is_file():
    precheck_errors.append("BEST_PT is unavailable. Run Cell 7 first.")

if not data_yaml_value or not Path(data_yaml_value).is_file():
    precheck_errors.append("DATA_YAML_PATH is unavailable. Run Cell 5 first.")

if not validation_records:
    precheck_errors.append(
        "Validation split is empty or unavailable. Run Cells 3–5 first."
    )

if not CLASS_NAMES_RUNTIME:
    precheck_errors.append("CLASS_NAMES is unavailable. Run Cell 1 first.")

if validation_records:
    if any(
        record.get("original_record_status") == "empty_negative_label"
        for record in validation_records
    ):
        precheck_errors.append("Validation contains empty-label records.")

    if any(
        record.get("parsed_id") in BAD_IDS_RUNTIME
        for record in validation_records
    ):
        precheck_errors.append("Validation contains BAD_IDS.")

    if any(
        record.get("parsed_id") in TRAIN_ONLY_RANGE_RUNTIME
        for record in validation_records
    ):
        precheck_errors.append("Validation contains train-only IDs.")

    if any(
        record.get("original_record_status") in {"missing_label", "invalid_label"}
        for record in validation_records
    ):
        precheck_errors.append("Validation contains unusable records.")

if precheck_errors:
    warnings.extend(precheck_errors)

    print("Validation skipped because prerequisite checks failed.")

    for message in precheck_errors:
        print(f"WARNING: {message}")

else:
    print("Validation started.")

    try:
        with quiet_execution(FRAMEWORK_LOG_DIR, "validation") as log_path:
            VALIDATION_LOG_PATH = log_path

            from ultralytics import YOLO

            validation_model = YOLO(str(best_pt_value))

            VAL_RESULTS = validation_model.val(
                data=str(data_yaml_value),
                split="val",
                imgsz=IMG_SIZE,
                batch=BATCH_SIZE,
                workers=WORKERS,
                project=str(OUTPUT_ROOT / "runs"),
                name=VALIDATION_RUN_NAME,
                exist_ok=True,
                plots=True,
                verbose=False,
            )

    except Exception as exc:
        errors.append(f"Validation execution failed: {exc}")
        errors.append(traceback.format_exc())

        print(
            "WARNING: Validation failed, but Cell 8 will continue "
            "and export available diagnostics."
        )


# ------------------------------------------------------------------
# Resolve possible Ultralytics output folders.
# ------------------------------------------------------------------
candidate_dirs: list[Path] = []

add_dir(candidate_dirs, VAL_RUN_DIR)
add_dir(candidate_dirs, getattr(VAL_RESULTS, "save_dir", None))
add_dir(
    candidate_dirs,
    getattr(getattr(validation_model, "validator", None), "save_dir", None),
)
add_dir(candidate_dirs, globals().get("RUN_DIR"))

if VAL_RESULTS is not None:
    actual_save_dir = getattr(VAL_RESULTS, "save_dir", None)
    validator_save_dir = getattr(
        getattr(validation_model, "validator", None),
        "save_dir",
        None,
    )

    if actual_save_dir:
        VAL_RUN_DIR = Path(actual_save_dir)

    elif validator_save_dir:
        VAL_RUN_DIR = Path(validator_save_dir)

    add_dir(candidate_dirs, VAL_RUN_DIR)


# ------------------------------------------------------------------
# Ask actual Ultralytics objects to export plots again if necessary.
# ------------------------------------------------------------------
if VAL_RESULTS is not None:
    try:
        VAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

        if hasattr(VAL_RESULTS, "plot"):
            try:
                VAL_RESULTS.plot(
                    save_dir=VAL_RUN_DIR,
                    names=CLASS_NAME_MAP,
                )
            except TypeError:
                VAL_RESULTS.plot(save_dir=VAL_RUN_DIR)

        confusion_matrix = getattr(VAL_RESULTS, "confusion_matrix", None)

        if confusion_matrix is not None and hasattr(confusion_matrix, "plot"):
            try:
                confusion_matrix.plot(
                    save_dir=VAL_RUN_DIR,
                    names=list(CLASS_NAME_MAP.values()),
                    normalize=False,
                )

                confusion_matrix.plot(
                    save_dir=VAL_RUN_DIR,
                    names=list(CLASS_NAME_MAP.values()),
                    normalize=True,
                )

            except Exception as exc:
                warnings.append(
                    f"Could not explicitly export confusion matrices: {exc}"
                )

    except Exception as exc:
        warnings.append(f"Could not request framework plot export: {exc}")


# ------------------------------------------------------------------
# Save real validation metrics and CSV summary.
# ------------------------------------------------------------------
raw_metrics = (
    getattr(VAL_RESULTS, "results_dict", {})
    if VAL_RESULTS is not None
    else {}
)

metrics = raw_metrics if isinstance(raw_metrics, dict) else {}

try:
    metrics = json.loads(json.dumps(metrics, default=str))
except Exception as exc:
    warnings.append(f"Could not serialize validation metrics: {exc}")
    metrics = {}

validation_class_counts = count_objects(validation_records)

validation_class_ids_present = sorted(
    {
        int(class_id)
        for record in validation_records
        for class_id in record.get("class_ids_present", [])
    }
)

summary_values = {
    "Precision": get_metric(
        metrics,
        ["metrics/precision(B)", "precision"],
    ),
    "Recall": get_metric(
        metrics,
        ["metrics/recall(B)", "recall"],
    ),
    "mAP@0.50": get_metric(
        metrics,
        ["metrics/mAP50(B)", "mAP50"],
    ),
    "mAP@0.50:0.95": get_metric(
        metrics,
        ["metrics/mAP50-95(B)", "mAP50-95"],
    ),
}

validation_metrics_path = OUTPUT_ROOT / "validation_metrics.json"
validation_summary_path = OUTPUT_ROOT / "validation_summary.csv"

safe_json(
    validation_metrics_path,
    {
        "metrics": metrics,
        "nc": NUM_CLASSES_RUNTIME,
        "class_names": CLASS_NAME_MAP,
        "validation_object_instances_per_class": validation_class_counts,
        "validation_class_ids_present": validation_class_ids_present,
        "validation_executed": VAL_RESULTS is not None,
        "warnings": warnings,
        "errors": errors,
    },
)

try:
    summary_rows = [
        {
            "row_type": "metric",
            "metric": metric_name,
            "value": metric_value,
            "class_id": "",
            "class_name": "",
            "instances": "",
        }
        for metric_name, metric_value in summary_values.items()
    ]

    summary_rows.extend(
        {
            "row_type": "validation_class_instances",
            "metric": "object_instances",
            "value": "",
            "class_id": class_id,
            "class_name": class_name,
            "instances": validation_class_counts.get(class_name, 0),
        }
        for class_id, class_name in enumerate(CLASS_NAMES_RUNTIME)
    )

    pd.DataFrame(summary_rows).to_csv(
        validation_summary_path,
        index=False,
        encoding="utf-8-sig",
    )

except Exception as exc:
    errors.append(f"Could not export validation_summary.csv: {exc}")


# ------------------------------------------------------------------
# Export artifacts. Missing artifacts generate warnings, never raises.
# ------------------------------------------------------------------
required_artifacts = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
]

artifact_status: dict[str, dict[str, str]] = {}

for filename in required_artifacts:
    artifact_status[filename] = {
        "status": (
            "copied"
            if copy_artifact(filename, candidate_dirs)
            else "not_found_yet"
        ),
        "source": artifact_sources.get(filename, ""),
    }

# If the framework did not write the curves, use real validation arrays.
render_missing_real_curves(VAL_RESULTS)

# Search again after plot export and real-array curve export.
for filename in required_artifacts:
    target = OUTPUT_ROOT / filename

    if target.is_file():
        if filename in artifact_generation:
            artifact_status[filename] = {
                "status": "generated_from_real_metrics",
                "source": artifact_generation[filename],
            }

        elif artifact_status[filename]["status"] == "not_found_yet":
            artifact_status[filename] = {
                "status": "available",
                "source": str(target),
            }

        continue

    if copy_artifact(filename, candidate_dirs):
        artifact_status[filename] = {
            "status": "copied",
            "source": artifact_sources.get(filename, ""),
        }

    else:
        artifact_status[filename] = {
            "status": "missing",
            "source": "",
        }

        warnings.append(
            f"{filename} is unavailable. The notebook will continue; "
            "see validation_artifact_report.json."
        )


# ------------------------------------------------------------------
# Always create diagnostics and metadata.
# ------------------------------------------------------------------
artifact_report_path = OUTPUT_ROOT / "validation_artifact_report.json"

safe_json(
    artifact_report_path,
    {
        "validation_started_at": VALIDATION_STARTED_AT.isoformat(
            timespec="seconds"
        ),
        "validation_run_dir": str(VAL_RUN_DIR),
        "validation_log": str(VALIDATION_LOG_PATH),
        "validation_executed": VAL_RESULTS is not None,
        "candidate_artifact_directories": [
            str(path) for path in candidate_dirs
        ],
        "artifact_status": artifact_status,
        "artifact_sources": artifact_sources,
        "artifact_generation": artifact_generation,
        "warnings": warnings,
        "errors": errors,
    },
)

safe_json(
    OUTPUT_ROOT / "validation_run_metadata.json",
    {
        "validation_run_dir": str(VAL_RUN_DIR),
        "validation_log": str(VALIDATION_LOG_PATH),
        "validation_metrics_json": str(validation_metrics_path),
        "validation_summary_csv": str(validation_summary_path),
        "validation_artifact_report_json": str(artifact_report_path),
        "validated_split": "val",
        "validation_executed": VAL_RESULTS is not None,
        "warnings_count": len(warnings),
        "errors_count": len(errors),
    },
)

if VAL_RESULTS is not None and not errors:
    print("Validation completed.")
else:
    print("Validation finished with warnings/errors; continuing to the next cell.")

print(f"Raw validation log: {VALIDATION_LOG_PATH}")

for name, value in summary_values.items():
    if value is not None:
        print(f"{name}: {value:.4f}")

missing_artifacts = [
    filename
    for filename, item in artifact_status.items()
    if item["status"] == "missing"
]

if missing_artifacts:
    print("WARNING: Missing artifacts:", ", ".join(missing_artifacts))
    print(f"Diagnostic report: {artifact_report_path}")
else:
    print("All required validation artifacts are available.")

print("Metrics exported.")


Validation started.
Validation completed.
Raw validation log: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/framework_logs/20260630_053824_validation.log
Precision: 0.9319
Recall: 0.9373
mAP@0.50: 0.9574
mAP@0.50:0.95: 0.7376
All required validation artifacts are available.
Metrics exported.


## 9. Sample Inference

In [9]:
# Cell 9 - Run concise sample inference, preferring validation positives
if "BEST_PT" not in globals() or not Path(BEST_PT).exists():
    raise FileNotFoundError("BEST_PT is unavailable. Run training and validation first.")
sample_records = [record for record in SPLITS["val"] if record["object_count"] > 0]
if not sample_records:
    sample_records = [record for record in SPLITS["train"] if record["object_count"] > 0]
if not sample_records:
    sample_records = list(SPLITS["train"])
if not sample_records:
    raise RuntimeError("No images are available for sample inference.")
sample_records = sorted(sample_records, key=lambda record: stable_score(record, RANDOM_SEED))[:SAMPLE_PREDICTION_COUNT]
sample_sources = [Path(record["source_image_path"]) for record in sample_records]
print("Sample inference started.")
with quiet_execution(FRAMEWORK_LOG_DIR, "sample_inference") as INFERENCE_LOG_PATH:
    from ultralytics import YOLO
    inference_model = YOLO(str(BEST_PT))
    PREDICTION_RESULTS = inference_model.predict(source=[str(path) for path in sample_sources], conf=0.25, imgsz=IMG_SIZE, save=True, project=str(OUTPUT_ROOT), name="sample_predictions", exist_ok=True, verbose=False)
SAMPLE_PREDICTIONS_DIR = Path(getattr(PREDICTION_RESULTS[0], "save_dir", OUTPUT_ROOT / "sample_predictions")) if PREDICTION_RESULTS else OUTPUT_ROOT / "sample_predictions"
for result in PREDICTION_RESULTS:
    source_name = Path(getattr(result, "path", "image")).name
    box_count = 0 if result.boxes is None else len(result.boxes)
    print(f"{source_name}: {box_count} detections")
    if result.boxes is None:
        continue
    for box in result.boxes[:10]:
        class_id = int(box.cls.detach().cpu().item())
        confidence = float(box.conf.detach().cpu().item())
        class_name = result.names.get(class_id, str(class_id))
        print(f"  {class_name}: {confidence:.3f}")
print(f"Sample predictions saved to: {SAMPLE_PREDICTIONS_DIR}")
print(f"Raw inference log: {INFERENCE_LOG_PATH}")


Sample inference started.
image0.jpg: 3 detections
  quan_tay_dai_den: 0.890
  ao_so_mi_trang: 0.885
  khan_quang_do: 0.878
image1.jpg: 3 detections
  ao_doan_thanh_nien: 0.921
  quan_dai_trang: 0.899
  khan_quang_do: 0.845
image2.jpg: 2 detections
  quan_tay_dai_den: 0.912
  ao_doan_thanh_nien: 0.881
image3.jpg: 1 detections
  quan_dai_trang: 0.918
image4.jpg: 3 detections
  ao_so_mi_trang: 0.748
  quan_dai_trang: 0.671
  ao_so_mi_trang: 0.452
image5.jpg: 4 detections
  quan_short_tay_den: 0.928
  ao_so_mi_trang: 0.817
  khan_quang_do: 0.404
  khan_quang_do: 0.340
image6.jpg: 2 detections
  ao_doan_thanh_nien: 0.908
  quan_tay_dai_den: 0.874
image7.jpg: 3 detections
  ao_so_mi_trang: 0.907
  khan_quang_do: 0.820
  quan_tay_dai_den: 0.560
Sample predictions saved to: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/sample_predictions
Raw inference log: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/framework_logs/20260630_053858_sample_inference.log


## 10. Windows 11 Deployment Package

In [11]:
# Cell 10 - Create and download the Windows 11 deployment package (resilient)
# Replace the entire old Cell 10 with this code.
# It automatically restores best.pt and last.pt from Google Drive when possible.
# Missing last.pt and non-essential charts/reports are warnings, not fatal errors.

from __future__ import annotations

import hashlib
import json
import shutil
import traceback
import zipfile
from datetime import datetime
from pathlib import Path

import yaml

# =============================================================================
# 1. Restore runtime values after Colab reconnects
# =============================================================================
OUTPUT_ROOT = Path(
    globals().get(
        "OUTPUT_ROOT",
        "/content/drive/MyDrive/DATN2/yolov8_uniform_training_output",
    )
)
ZIP_PATH = Path(
    globals().get(
        "ZIP_PATH",
        "/content/drive/MyDrive/DATN2/yolov8_uniform_windows_package.zip",
    )
)
WORK_DIR = Path(globals().get("WORK_DIR", "/content/uniform_yolo_dataset"))
DATA_YAML_PATH = Path(globals().get("DATA_YAML_PATH", WORK_DIR / "data.yaml"))
MODEL_WEIGHTS = str(globals().get("MODEL_WEIGHTS", "yolov8s.pt"))

DEFAULT_CLASS_NAMES = [
    "ao_so_mi_trang",
    "ao_doan_thanh_nien",
    "quan_tay_dai_den",
    "khan_quang_do",
    "quan_short_tay_den",
    "quan_dai_trang",
]
CLASS_NAMES = list(globals().get("CLASS_NAMES", DEFAULT_CLASS_NAMES))
if not CLASS_NAMES:
    CLASS_NAMES = DEFAULT_CLASS_NAMES.copy()
NUM_CLASSES = len(CLASS_NAMES)

PACKAGE_ROOT = OUTPUT_ROOT / "windows_package_tmp"
PACKAGE_REPORT_PATH = OUTPUT_ROOT / "windows_package_build_report.json"

warnings_list: list[str] = []
errors_list: list[str] = []
package_manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "target_system": "Windows 11 64-bit",
    "model_weights": MODEL_WEIGHTS,
    "training_method": "transfer learning / fine-tuning from yolov8s.pt",
    "nc": NUM_CLASSES,
    "class_names": {index: name for index, name in enumerate(CLASS_NAMES)},
    "files": [],
    "warnings": warnings_list,
}


def safe_write_json(path: Path, payload) -> bool:
    try:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(
            json.dumps(payload, ensure_ascii=False, indent=2, default=str),
            encoding="utf-8",
        )
        return True
    except Exception as exc:
        errors_list.append(f"Could not write {path.name}: {exc}")
        return False


def valid_file(value) -> bool:
    try:
        path = Path(value)
        return path.is_file() and path.stat().st_size > 0
    except Exception:
        return False


def add_to_manifest(relative_path: str) -> None:
    relative_path = relative_path.replace("\\", "/")
    if relative_path not in package_manifest["files"]:
        package_manifest["files"].append(relative_path)


def latest_matching_file(filename: str) -> Path | None:
    matches: list[Path] = []
    for root in (OUTPUT_ROOT / "runs", OUTPUT_ROOT):
        if not root.exists():
            continue
        try:
            for path in root.rglob(filename):
                if (
                    path.is_file()
                    and path.stat().st_size > 0
                    and "windows_package_tmp" not in path.parts
                ):
                    matches.append(path)
        except Exception as exc:
            warnings_list.append(f"Could not search {root} for {filename}: {exc}")

    if not matches:
        return None

    try:
        return max(matches, key=lambda item: item.stat().st_mtime)
    except Exception as exc:
        warnings_list.append(f"Could not choose the latest {filename}: {exc}")
        return None


def restore_checkpoint(variable_name: str, filename: str) -> Path | None:
    current_value = globals().get(variable_name)
    if valid_file(current_value):
        return Path(current_value)

    found = latest_matching_file(filename)
    if found is not None:
        warnings_list.append(f"{variable_name} restored automatically: {found}")
    return found


def restore_last_checkpoint(best_path: Path) -> Path | None:
    current_value = globals().get("LAST_PT")
    if valid_file(current_value):
        return Path(current_value)

    same_weights_folder = best_path.parent / "last.pt"
    if valid_file(same_weights_folder):
        warnings_list.append(
            f"LAST_PT restored from the same weights folder: {same_weights_folder}"
        )
        return same_weights_folder

    return restore_checkpoint("LAST_PT", "last.pt")


def copy_into_package(source, relative_target: str) -> bool:
    if source is None:
        return False

    try:
        source = Path(source)
        if not source.is_file():
            return False

        destination = PACKAGE_ROOT / relative_target
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        add_to_manifest(relative_target)
        return True
    except Exception as exc:
        warnings_list.append(
            f"Could not copy {source} to package path {relative_target}: {exc}"
        )
        return False


def find_artifact(filename: str, preferred_dirs: list[Path]) -> Path | None:
    checked: set[Path] = set()
    for directory in preferred_dirs:
        try:
            candidate = Path(directory) / filename
        except Exception:
            continue
        if candidate in checked:
            continue
        checked.add(candidate)
        if candidate.is_file() and candidate.stat().st_size > 0:
            return candidate
    return latest_matching_file(filename)


def copy_optional_artifact(filename: str, preferred_dirs: list[Path]) -> None:
    source = find_artifact(filename, preferred_dirs)
    if not copy_into_package(source, filename):
        warnings_list.append(f"Optional artifact unavailable: {filename}")


def sha256_file(path: Path) -> str | None:
    try:
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()
    except Exception as exc:
        warnings_list.append(f"Could not calculate SHA-256 for {path.name}: {exc}")
        return None


# =============================================================================
# 2. Restore best.pt and last.pt. Missing last.pt does not stop this cell.
# =============================================================================
BEST_PT = restore_checkpoint("BEST_PT", "best.pt")

if BEST_PT is None:
    errors_list.append(
        "No valid best.pt was found under OUTPUT_ROOT. The deployment package cannot be created without best.pt."
    )
    safe_write_json(
        PACKAGE_REPORT_PATH,
        {
            "package_created": False,
            "zip_path": str(ZIP_PATH),
            "best_pt": "",
            "last_pt": "",
            "warnings": warnings_list,
            "errors": errors_list,
        },
    )
    print("WARNING: Windows package was not created because best.pt was not found.")
    print(f"Diagnostic report: {PACKAGE_REPORT_PATH}")

else:
    LAST_PT = restore_last_checkpoint(BEST_PT)
    RUN_DIR = BEST_PT.parent.parent if BEST_PT.parent.name == "weights" else BEST_PT.parent

    checkpoint_status = {
        "best_pt": str(BEST_PT),
        "best_pt_included": True,
        "last_pt": str(LAST_PT) if LAST_PT else "",
        "last_pt_included": LAST_PT is not None,
        "note": (
            "Both best.pt and last.pt were included."
            if LAST_PT is not None
            else (
                "last.pt was unavailable. The package contains best.pt only; "
                "best.pt is sufficient and recommended for inference."
            )
        ),
    }

    if LAST_PT is None:
        warnings_list.append(
            "last.pt was not found. The package will include best.pt only. "
            "This does not prevent Windows inference."
        )

    try:
        # =========================================================================
        # 3. Prepare a fresh temporary package folder
        # =========================================================================
        if PACKAGE_ROOT.exists():
            try:
                shutil.rmtree(PACKAGE_ROOT)
            except Exception as exc:
                warnings_list.append(
                    f"Could not remove old package folder {PACKAGE_ROOT}: {exc}"
                )

        PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)

        if not copy_into_package(BEST_PT, "best.pt"):
            errors_list.append("best.pt could not be copied into the package.")

        if LAST_PT is not None:
            if not copy_into_package(LAST_PT, "last.pt"):
                warnings_list.append("last.pt was found but could not be copied.")

        # Build deployment metadata YAML with no test key.
        package_yaml = {
            "path": ".",
            "train": "images/train",
            "val": "images/val",
            "nc": NUM_CLASSES,
            "names": {index: name for index, name in enumerate(CLASS_NAMES)},
        }
        if DATA_YAML_PATH.is_file():
            try:
                original_yaml = yaml.safe_load(
                    DATA_YAML_PATH.read_text(encoding="utf-8")
                ) or {}
                if isinstance(original_yaml, dict):
                    for key in ("path", "train", "val"):
                        if key in original_yaml:
                            package_yaml[key] = original_yaml[key]
            except Exception as exc:
                warnings_list.append(
                    f"Could not read temporary data.yaml; safe metadata YAML was used: {exc}"
                )
        else:
            warnings_list.append(
                "Temporary data.yaml is unavailable; safe metadata YAML was generated."
            )

        package_yaml.pop("test", None)
        package_yaml["nc"] = NUM_CLASSES
        package_yaml["names"] = {
            index: name for index, name in enumerate(CLASS_NAMES)
        }

        (PACKAGE_ROOT / "data.yaml").write_text(
            yaml.safe_dump(package_yaml, sort_keys=False, allow_unicode=True),
            encoding="utf-8",
        )
        add_to_manifest("data.yaml")

        (PACKAGE_ROOT / "classes.txt").write_text(
            "\n".join(CLASS_NAMES) + "\n", encoding="utf-8"
        )
        add_to_manifest("classes.txt")

        (PACKAGE_ROOT / "class_names.json").write_text(
            json.dumps(
                {index: name for index, name in enumerate(CLASS_NAMES)},
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
        add_to_manifest("class_names.json")

        safe_write_json(
            PACKAGE_ROOT / "model_provenance.json",
            {
                "deployment_model": "best.pt",
                "base_model_weights": MODEL_WEIGHTS,
                "framework": "Ultralytics YOLOv8",
                "training_method": "transfer learning / fine-tuning",
                "provenance_statement": (
                    "The detector was fine-tuned from yolov8s.pt using the "
                    "Ultralytics YOLOv8 framework; it was not implemented from scratch."
                ),
                "nc": NUM_CLASSES,
                "class_names": {
                    index: name for index, name in enumerate(CLASS_NAMES)
                },
                "checkpoint_status": checkpoint_status,
            },
        )
        add_to_manifest("model_provenance.json")
        safe_write_json(PACKAGE_ROOT / "checkpoint_status.json", checkpoint_status)
        add_to_manifest("checkpoint_status.json")

        # =========================================================================
        # 4. Create Windows inference files and README
        # =========================================================================
        infer_windows_py = f'''from __future__ import annotations

import argparse
from pathlib import Path

from ultralytics import YOLO

IMAGE_EXTENSIONS = {{".jpg", ".jpeg", ".png", ".bmp", ".webp"}}
VIDEO_EXTENSIONS = {{".mp4", ".avi", ".mov", ".mkv", ".wmv", ".m4v"}}
CLASS_NAMES = {CLASS_NAMES!r}


def parse_source(value: str):
    value = value.strip()
    if value.isdigit():
        return int(value)

    source_path = Path(value).expanduser()
    if not source_path.exists():
        raise FileNotFoundError(f"Source path not found: {{source_path}}")

    allowed_extensions = IMAGE_EXTENSIONS | VIDEO_EXTENSIONS
    if source_path.is_file() and source_path.suffix.lower() not in allowed_extensions:
        raise ValueError(f"Unsupported source extension: {{source_path.suffix}}")

    return str(source_path)


def main() -> int:
    parser = argparse.ArgumentParser(
        description="Run school-uniform YOLOv8 inference on Windows 11 64-bit."
    )
    parser.add_argument(
        "--source",
        required=True,
        help="Image, folder, video, or webcam index such as 0.",
    )
    parser.add_argument(
        "--weights",
        default="best.pt",
        help="Weights path. Default: best.pt next to this script.",
    )
    parser.add_argument("--conf", type=float, default=0.25)
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument(
        "--output",
        default="runs_uniform_predict",
        help="Folder for annotated outputs.",
    )
    args = parser.parse_args()

    package_dir = Path(__file__).resolve().parent
    weights_path = Path(args.weights).expanduser()
    if not weights_path.is_absolute():
        weights_path = package_dir / weights_path
    if not weights_path.is_file():
        raise FileNotFoundError(f"Weights not found: {{weights_path}}")

    model = YOLO(str(weights_path))
    results = model.predict(
        source=parse_source(args.source),
        conf=args.conf,
        imgsz=args.imgsz,
        save=True,
        project=str(Path(args.output).expanduser()),
        name="predict",
        exist_ok=True,
        verbose=False,
    )

    for result in results:
        source_name = Path(getattr(result, "path", str(args.source))).name
        box_count = 0 if result.boxes is None else len(result.boxes)
        print(f"{{source_name}}: {{box_count}} detections")
        if result.boxes is None:
            continue

        for box in result.boxes:
            class_id = int(box.cls.detach().cpu().item())
            confidence = float(box.conf.detach().cpu().item())
            class_name = result.names.get(
                class_id,
                CLASS_NAMES[class_id] if 0 <= class_id < len(CLASS_NAMES) else str(class_id),
            )
            xyxy = [round(float(value), 2) for value in box.xyxy[0].detach().cpu().tolist()]
            print(f"  {{class_name}}: conf={{confidence:.3f}}, box_xyxy={{xyxy}}")

    if results:
        print(f"Annotated outputs saved to: {{results[0].save_dir}}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

        requirements_windows_txt = """ultralytics
opencv-python
pillow
numpy
"""
        run_example_windows_bat = r"""@echo off
REM Image inference
python infer_windows.py --source demo_image.jpg --weights best.pt --conf 0.25

REM Folder inference
python infer_windows.py --source demo_images --weights best.pt --conf 0.25

REM Video inference
python infer_windows.py --source demo_video.mp4 --weights best.pt --conf 0.25

REM Webcam inference
python infer_windows.py --source 0 --weights best.pt --conf 0.25

pause
"""

        class_mapping_text = "\n".join(
            f"{index}: {name}" for index, name in enumerate(CLASS_NAMES)
        )
        last_checkpoint_text = (
            "`last.pt` is included and can be used for continuation or audit."
            if LAST_PT is not None
            else (
                "`last.pt` was unavailable, so this recovered package intentionally "
                "contains `best.pt` only. `best.pt` is sufficient and recommended for inference."
            )
        )
        readme_windows_md = f"""# School Uniform YOLOv8 Detector - Windows 11 64-bit

## Supported System
Windows 11 64-bit.

## Python
Use Python 3.10 or Python 3.11 64-bit.

## Create and Activate a Virtual Environment
```bat
python -m venv .venv
.venv\\Scripts\\activate
```

## Install Dependencies
```bat
python -m pip install --upgrade pip
pip install -r requirements_windows.txt
```

## Image Inference
```bat
python infer_windows.py --source demo_image.jpg --weights best.pt --conf 0.25
```

## Folder Inference
```bat
python infer_windows.py --source demo_images --weights best.pt --conf 0.25
```

## Video Inference
```bat
python infer_windows.py --source demo_video.mp4 --weights best.pt --conf 0.25
```

## Webcam Inference
```bat
python infer_windows.py --source 0 --weights best.pt --conf 0.25
```

## Default Output Folder
Annotated outputs are saved under `runs_uniform_predict/predict`, unless `--output` is changed.

## Class Mapping
{class_mapping_text}

## Model Files
`best.pt` is the checkpoint with the best validation fitness and is recommended for inference.

{last_checkpoint_text}

## Model Provenance
The detector was fine-tuned from `yolov8s.pt` using the Ultralytics YOLOv8 framework. It was not implemented from scratch.

## Package Notes
- `data.yaml` contains only train/validation metadata and has no test key.
- Raw source images and original source labels are intentionally excluded.
- See `checkpoint_status.json` and `windows_package_build_report.json` for diagnostics.
"""

        (PACKAGE_ROOT / "infer_windows.py").write_text(infer_windows_py, encoding="utf-8")
        (PACKAGE_ROOT / "requirements_windows.txt").write_text(requirements_windows_txt, encoding="utf-8")
        (PACKAGE_ROOT / "run_example_windows.bat").write_text(run_example_windows_bat, encoding="utf-8")
        (PACKAGE_ROOT / "README_WINDOWS_11.md").write_text(readme_windows_md, encoding="utf-8")
        for file_name in (
            "infer_windows.py",
            "requirements_windows.txt",
            "run_example_windows.bat",
            "README_WINDOWS_11.md",
        ):
            add_to_manifest(file_name)

        # =========================================================================
        # 5. Add real reports and visual artifacts when available (all optional)
        # =========================================================================
        validation_run_dir = Path(
            globals().get(
                "VAL_RUN_DIR",
                OUTPUT_ROOT / "runs" / str(
                    globals().get("VALIDATION_RUN_NAME", "uniform_detector_validation")
                ),
            )
        )
        preferred_dirs = [
            OUTPUT_ROOT,
            RUN_DIR,
            validation_run_dir,
            RUN_DIR / "weights",
            validation_run_dir / "weights",
        ]

        artifact_names = [
            "results.csv",
            "validation_metrics.json",
            "validation_summary.csv",
            "validation_artifact_report.json",
            "confusion_matrix.png",
            "confusion_matrix_normalized.png",
            "PR_curve.png",
            "F1_curve.png",
            "P_curve.png",
            "R_curve.png",
            "training_curves.png",
            "class_distribution.png",
            "class_distribution.csv",
            "split_summary.csv",
            "split_manifest.csv",
            "dataset_validation_report.json",
            "dataset_validation_report.csv",
            "dataset_overview.json",
            "training_configuration.json",
            "training_run_metadata.json",
            "validation_run_metadata.json",
            "training_args.yaml",
        ]
        for artifact_name in artifact_names:
            copy_optional_artifact(artifact_name, preferred_dirs)

        # Copy only generated previews/logs, never raw dataset_images or labels/all.
        for optional_dir_name in (
            "sample_predictions",
            "bbox_preview_samples",
            "framework_logs",
        ):
            source_dir = OUTPUT_ROOT / optional_dir_name
            if not source_dir.is_dir():
                continue
            try:
                for file_path in sorted(source_dir.rglob("*")):
                    if file_path.is_file():
                        relative_target = str(
                            Path(optional_dir_name) / file_path.relative_to(source_dir)
                        )
                        copy_into_package(file_path, relative_target)
            except Exception as exc:
                warnings_list.append(
                    f"Could not package optional directory {optional_dir_name}: {exc}"
                )

        # =========================================================================
        # 6. Write manifests, checksums, diagnostics, and ZIP
        # =========================================================================
        package_report = {
            "package_created": False,
            "zip_path": str(ZIP_PATH),
            "package_root": str(PACKAGE_ROOT),
            "best_pt": str(BEST_PT),
            "last_pt": str(LAST_PT) if LAST_PT else "",
            "checkpoint_status": checkpoint_status,
            "warnings": warnings_list,
            "errors": errors_list,
        }

        safe_write_json(PACKAGE_ROOT / "windows_package_build_report.json", package_report)
        add_to_manifest("windows_package_build_report.json")
        add_to_manifest("package_manifest.json")
        add_to_manifest("checksums.json")
        safe_write_json(PACKAGE_ROOT / "package_manifest.json", package_manifest)

        checksums = {}
        for file_path in sorted(PACKAGE_ROOT.rglob("*")):
            if file_path.is_file() and file_path.name != "checksums.json":
                digest = sha256_file(file_path)
                if digest is not None:
                    checksums[str(file_path.relative_to(PACKAGE_ROOT)).replace("\\", "/")] = digest
        safe_write_json(PACKAGE_ROOT / "checksums.json", checksums)

        try:
            if ZIP_PATH.exists():
                ZIP_PATH.unlink()
            ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zip_handle:
                for file_path in sorted(PACKAGE_ROOT.rglob("*")):
                    if file_path.is_file():
                        zip_handle.write(
                            file_path,
                            arcname=str(file_path.relative_to(PACKAGE_ROOT)).replace("\\", "/"),
                        )
        except Exception as exc:
            errors_list.append(f"Could not create ZIP archive: {exc}")
            errors_list.append(traceback.format_exc())

        zip_created = ZIP_PATH.is_file() and ZIP_PATH.stat().st_size > 0

        if zip_created:
            try:
                with zipfile.ZipFile(ZIP_PATH) as zip_handle:
                    zip_names = set(zip_handle.namelist())
                    package_data_yaml = yaml.safe_load(
                        zip_handle.read("data.yaml").decode("utf-8")
                    ) or {}

                essential_entries = {
                    "best.pt",
                    "data.yaml",
                    "classes.txt",
                    "class_names.json",
                    "infer_windows.py",
                    "requirements_windows.txt",
                    "run_example_windows.bat",
                    "README_WINDOWS_11.md",
                    "model_provenance.json",
                    "checkpoint_status.json",
                    "package_manifest.json",
                    "checksums.json",
                }
                missing_entries = sorted(essential_entries - zip_names)
                if missing_entries:
                    errors_list.append(f"ZIP is missing essential entries: {missing_entries}")

                if LAST_PT is not None and "last.pt" not in zip_names:
                    errors_list.append("last.pt was found but is missing from the ZIP.")

                if "test" in package_data_yaml:
                    errors_list.append("Packaged data.yaml contains a forbidden test key.")

                forbidden_paths = [
                    name
                    for name in zip_names
                    if (
                        name in {"test.txt", "test.csv"}
                        or "/test/" in f"/{name}"
                        or name.startswith("dataset_images/")
                        or name.startswith("labels/all/")
                    )
                ]
                if forbidden_paths:
                    errors_list.append(
                        f"Forbidden raw-dataset/test artifacts found in ZIP: {forbidden_paths}"
                    )
            except Exception as exc:
                errors_list.append(f"Could not validate ZIP content: {exc}")

        package_report.update(
            {
                "package_created": zip_created,
                "warnings": warnings_list,
                "errors": errors_list,
            }
        )
        safe_write_json(PACKAGE_REPORT_PATH, package_report)
        safe_write_json(PACKAGE_ROOT / "windows_package_build_report.json", package_report)

        # Recreate the ZIP once more so the final report is included inside it.
        if zip_created:
            try:
                with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zip_handle:
                    for file_path in sorted(PACKAGE_ROOT.rglob("*")):
                        if file_path.is_file():
                            zip_handle.write(
                                file_path,
                                arcname=str(file_path.relative_to(PACKAGE_ROOT)).replace("\\", "/"),
                            )
            except Exception as exc:
                warnings_list.append(f"Could not refresh ZIP with final report: {exc}")
                safe_write_json(PACKAGE_REPORT_PATH, package_report)

        if ZIP_PATH.is_file() and ZIP_PATH.stat().st_size > 0:
            print("Windows deployment package created.")
            print(f"Package path: {ZIP_PATH}")
            print(f"Package size MB: {ZIP_PATH.stat().st_size / (1024 ** 2):.2f}")
            print(f"BEST_PT included: {BEST_PT}")
            if LAST_PT is not None:
                print(f"LAST_PT included: {LAST_PT}")
            else:
                print("WARNING: last.pt was unavailable; the package contains best.pt only.")

            if warnings_list:
                print(f"Package warnings: {len(warnings_list)}")
                for warning in warnings_list[:10]:
                    print(f"WARNING: {warning}")
            if errors_list:
                print(f"Package validation notes: {len(errors_list)}")
                for error in errors_list[:10]:
                    print(f"NOTE: {error}")

            print(f"Build report: {PACKAGE_REPORT_PATH}")
            try:
                from google.colab import files
                files.download(str(ZIP_PATH))
            except Exception as exc:
                print(
                    "WARNING: ZIP was created but automatic browser download did not start: "
                    f"{exc}"
                )
        else:
            print("WARNING: ZIP package was not created successfully.")
            print(f"Build report: {PACKAGE_REPORT_PATH}")

    except Exception as exc:
        errors_list.append(f"Unexpected package build error: {exc}")
        errors_list.append(traceback.format_exc())
        safe_write_json(
            PACKAGE_REPORT_PATH,
            {
                "package_created": False,
                "zip_path": str(ZIP_PATH),
                "best_pt": str(BEST_PT),
                "last_pt": str(LAST_PT) if LAST_PT else "",
                "warnings": warnings_list,
                "errors": errors_list,
            },
        )
        print(
            "WARNING: The package build encountered an error, but this cell "
            "will not interrupt the remaining notebook workflow."
        )
        print(f"Diagnostic report: {PACKAGE_REPORT_PATH}")


Windows deployment package created.
Package path: /content/drive/MyDrive/DATN2/yolov8_uniform_windows_package.zip
Package size MB: 53.52
BEST_PT included: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/runs/uniform_detector_training/weights/best.pt
LAST_PT included: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/runs/uniform_detector_training/weights/last.pt
Package warnings: 1
Build report: /content/drive/MyDrive/DATN2/yolov8_uniform_training_output/windows_package_build_report.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>